# Ingestion & Chunking — the LangChain way

**Notebook 02 · Phase 2 (Data preparation)** · Stack: `langchain` + `langchain-openai`

Before a model can *retrieve* anything, two things have to go right:

1. **Ingestion & parsing** — turning PDFs, HTML, Word docs, and scanned images into
   clean text. *Most quality loss in a RAG system happens here, before chunking even
   starts.* A mangled table can never be retrieved correctly no matter how good your
   embeddings are.
2. **Chunking** — splitting that clean text into retrieval-sized units. We walk the
   six strategies **in order of maturity**, from naive fixed-size to retrieval-aware
   hierarchical.

> This notebook has a twin: **Notebook 03** does the *exact same thing* with
> **LlamaIndex**, so you can compare the two frameworks side by side on identical
> sample documents. Each section here ends with the LangChain API used; Notebook 03
> shows the LlamaIndex equivalent.

## 0. Install dependencies

Run this cell **first**. It installs everything this notebook needs into the *active
kernel* using `%pip`, so it works in any fresh environment — if a package is already
present, pip simply skips it. **After the very first install, restart the kernel**
(Kernel → Restart Kernel) so newly installed packages are picked up, then run the rest
top to bottom.

> `numpy` is pinned `<2` because the OCR stack (opencv / rapidocr) needs it.

In [1]:
# Install dependencies into the ACTIVE kernel (idempotent — skips what's already there).
%pip install -q \
    "numpy<2" \
    langchain langchain-core langchain-openai \
    langchain-community langchain-text-splitters langchain-experimental langchain-classic \
    langchain-pymupdf4llm rapidocr-onnxruntime pypdf docx2txt beautifulsoup4 lxml \
    tiktoken python-dotenv
print("✅ Dependencies ready. If this was a fresh install, restart the kernel now, then re-run.")


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: /Users/mohamednoordeenalaudeen/Documents/GenAI-2026/.venv/bin/python -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
✅ Dependencies ready. If this was a fresh install, restart the kernel now, then re-run.


## 1. Setup

Sample documents live in `./data/` (generated by `data/_generate_sample_data.py`).
API keys load from `../.env`. We use OpenAI embeddings for the retrieval-aware
strategies (3–5) — the same `OPENAI_API_KEY` from Notebook 01, nothing new required.

In [2]:
import warnings, os, re
warnings.filterwarnings("ignore")  # silence library sunset/deprecation notices for a clean teaching output
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(Path.cwd().parent / ".env")
DATA = Path.cwd() / "data"

from langchain_openai import OpenAIEmbeddings, ChatOpenAI
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

print("OPENAI_API_KEY set:", bool(os.getenv("OPENAI_API_KEY")))
print("sample files:", sorted(p.name for p in DATA.iterdir() if p.is_file() and not p.name.startswith("_")))

OPENAI_API_KEY set: True
sample files: ['eval_corpus.py', 'sample_contract.docx', 'sample_page.html', 'sample_report.pdf', 'sample_scanned.png']


### A helper to *see* chunks

`inspect_chunks()` prints how many chunks a strategy produced, their token
statistics, and a preview with visible `┃` boundaries — so the effect of each
strategy is tangible rather than abstract.

In [3]:
import tiktoken
_enc = tiktoken.get_encoding("cl100k_base")
def ntok(s: str) -> int:
    return len(_enc.encode(s))

def as_text(chunk):
    return chunk.page_content if hasattr(chunk, "page_content") else str(chunk)

def inspect_chunks(chunks, n_preview=3, width=200):
    texts = [as_text(c) for c in chunks]
    toks = [ntok(t) for t in texts] or [0]
    print(f"→ {len(texts)} chunks | tokens min/avg/max = {min(toks)}/{sum(toks)//len(toks)}/{max(toks)}")
    for i, t in enumerate(texts[:n_preview]):
        preview = " ".join(t.split())[:width]
        print(f"┃ [chunk {i}] {preview}{'…' if len(t) > width else ''}")

## 2. Ingestion & parsing — where quality is won or lost

We parse four formats and inspect what survives. The headline demo is the **PDF
table**: a naive text extractor flattens it into an ambiguous list of numbers, while
a layout-aware parser preserves the table structure.

### 2a. PDF — naive vs. layout-aware (the money demo)

`PyPDFLoader` extracts raw text; `PyMuPDF4LLMLoader` reconstructs layout and emits
Markdown, keeping the table intact.

In [4]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_pymupdf4llm import PyMuPDF4LLMLoader

naive = PyPDFLoader(str(DATA / "sample_report.pdf")).load()[0].page_content
aware = PyMuPDF4LLMLoader(str(DATA / "sample_report.pdf"), table_strategy="lines").load()[0].page_content

def table_region(text):
    i = text.find("Region")
    return text[i:i + 320]

print("=== PyPDFLoader (naive) — the table becomes an ambiguous list ===")
print(table_region(naive))
print("\n=== PyMuPDF4LLMLoader (layout-aware) — the table survives as Markdown ===")
print(table_region(aware))

=== Document parser messages ===


Using RapidOCR and Tesseract for OCR processing.



=== PyPDFLoader (naive) — the table becomes an ambiguous list ===
Regional performance is broken out in the table below.
Region
Shipments
On-Time %
Revenue ($M)
Northwest
128,400
97.2%
14.6
West
204,910
98.1%
23.9
South
176,220
96.8%
19.4
Northeast
92,180
89.3%
11.2
Total
601,710
95.9%
69.1
Action items: (1) complete the Northeast warehouse migration by July 15; (2) reallocate two de

=== PyMuPDF4LLMLoader (layout-aware) — the table survives as Markdown ===
Regional performance is broken out in the table below. 

|**Region**|**Shipments**|**On-Time %**|**Revenue ($M)**|
|---|---|---|---|
|Northwest|128,400|97.2%|14.6|
|West|204,910|98.1%|23.9|
|South|176,220|96.8%|19.4|
|Northeast|92,180|89.3%|11.2|
|**Total**|**601,710**|**95.9%**|**69.1**|



Action items: (1) complete 


**Look at the difference.** In the naive output you cannot tell that `128,400` is
Northwest's *shipments* and `97.2%` is its *on-time rate* — rows and columns are
gone. The layout-aware Markdown preserves the grid, so a retrieved chunk still means
something. **This is the single highest-leverage decision in ingestion.**

### 2b. HTML and Word

Structured formats need format-aware loaders too.

In [5]:
from langchain_community.document_loaders import BSHTMLLoader, Docx2txtLoader

html_text = BSHTMLLoader(str(DATA / "sample_page.html")).load()[0].page_content
docx_text = Docx2txtLoader(str(DATA / "sample_contract.docx")).load()[0].page_content

print("=== HTML (BSHTMLLoader) — first 200 chars ===")
print(" ".join(html_text.split())[:200], "…\n")
print("=== Word (Docx2txtLoader) — clause 5 ===")
i = docx_text.find("Remote-Work")
print(docx_text[i:i + 240])

=== HTML (BSHTMLLoader) — first 200 chars ===
Northwind Logistics — Support FAQ Northwind Logistics — Support FAQ Answers to common questions about shipping, tracking, and returns. Shipping Standard shipping takes 3–5 business days. Expedited shi …

=== Word (Docx2txtLoader) — clause 5 ===
Remote-Work Reimbursement (HR-2024-17 §7.3)

Employees working remotely at least three (3) days per week are eligible for a home-internet stipend of $45 per month, reimbursed via the monthly expense report. Contractors are not eligible.

6.


### 2c. Scanned image — OCR

A scanned memo is just pixels. OCR recovers the text — imperfectly. Notice the small
artifacts (`INTERNALMEMO`, `causedby`): OCR noise is real and flows downstream, which
is why an OCR quality gate matters before you ever chunk.

In [6]:
from rapidocr_onnxruntime import RapidOCR

_ocr = RapidOCR()
result, _ = _ocr(str(DATA / "sample_scanned.png"))
ocr_text = "\n".join(line[1] for line in result) if result else "(no text found)"
print("=== OCR (RapidOCR) — recovered text ===")
print(ocr_text)

=== OCR (RapidOCR) — recovered text ===
INTERNALMEMO
TO：
All warehouse staff
FRoM: Operations, Northwind Logistics
DATE: 2024-06-14
RE：
Incident INC-5521
The claims-search service had a 22-minute outage on
2024-06-14 causedby a Qdrant snapshot restorethat
exhausted disk.Fix: volume size increased and a disk-
usage alert added at 80%. Owner: platform team.


### Industry practice: OCR is a tiered pipeline, not one tool

Production teams rarely bet on a single OCR engine. Most run a cheap, fast pass first
and only escalate the documents that need more:

- **Tier 1 &mdash; free/local.** Tesseract or PaddleOCR/RapidOCR. Handles the 80-90% of
  clean, well-scanned pages. ~1-2 seconds a page, no API key, runs anywhere.
- **Tier 2 &mdash; cloud document AI.** AWS Textract, Azure Document Intelligence, Google
  Document AI. Adds table structure, form fields, and key-value extraction that classical
  OCR can't reliably preserve. Roughly $1.50-$30 per 1,000 pages depending on features.
- **Tier 3 &mdash; vision-capable LLMs.** GPT-5/Gemini/Claude vision, or LlamaParse. Best
  accuracy on messy scans, handwriting, and complex layouts &mdash; but slow (commonly
  16-33 seconds per page) and the most expensive tier.

The actual pattern: run Tier 1 (or Tier 2 if the document has real structure), check a
confidence score, and only escalate low-confidence pages up to Tier 3. Nobody runs their
whole corpus through a vision LLM &mdash; it's an escalation path, not a default.

If you only remember one line: **Tesseract for free/local, Textract/Document Intelligence
for managed cloud, a vision LLM only for the hard minority that needs it.**


**Takeaway for Part 2:** garbage in, garbage out. Layout-aware PDF parsing, correct
HTML/Word extraction, and clean OCR determine the ceiling on retrieval quality. No
embedding model or reranker recovers information that parsing already destroyed.

## 3. Chunking — six strategies in order of maturity

A **chunk** is the atomic unit of retrieval: it is what gets embedded, indexed, and
handed to the model. Chunk too large and the embedding blurs several topics; chunk
too small and you lose the context needed to answer. We use the layout-aware report
(from Part 2) and the contract as our source text.

In [7]:
report_md = aware              # layout-aware Markdown from section 2a (tables preserved)
contract  = docx_text          # contract clauses from section 2b
print("report_md tokens:", ntok(report_md), "| contract tokens:", ntok(contract))

report_md tokens: 254 | contract tokens: 278


### Strategy 1 — Fixed-size / token-based  *(baseline, naive)*

Split every N tokens with optional overlap. Fast, deterministic, dependency-free —
but it cuts blindly through sentences and tables. Watch a chunk boundary land in the
middle of the table.

**LangChain API:** `TokenTextSplitter`

In [8]:
from langchain_text_splitters import TokenTextSplitter

fixed = TokenTextSplitter(chunk_size=80, chunk_overlap=10)
fixed_chunks = fixed.split_text(report_md)
inspect_chunks(fixed_chunks)

→ 4 chunks | tokens min/avg/max = 58/70/78
┃ [chunk 0] # **Northwind Logistics — Q2 2024 Operations Report** This report summarizes regional shipment performance for the second quarter of 2024. Overall shipment volume grew 8.4% quarter over quarter, drive…
┃ [chunk 1] throughput. Regional performance is broken out in the table below. |**Region**|**Shipments**|**On-Time %**|**Revenue ($M)**| |---|---|---|---| |Northwest|128,400|97.2%|14.6| |West|204,910|
┃ [chunk 2] 6| |West|204,910|98.1%|23.9| |South|176,220|96.8%|19.4| |Northeast|92,180|89.3%|11.2| |**Total**|**601,710**|**95.9%**|**69.1**|


### Strategy 2 — Recursive character/token splitting  *(sane default)*

Tries a priority list of separators (`\n\n` → `\n` → ` `) to break at the most
natural boundary that still fits the window. This is the de-facto default for
general text.

**LangChain API:** `RecursiveCharacterTextSplitter`

In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

recursive = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=60)
rec_chunks = recursive.split_text(report_md)
inspect_chunks(rec_chunks)

→ 3 chunks | tokens min/avg/max = 46/83/132
┃ [chunk 0] # **Northwind Logistics — Q2 2024 Operations Report** This report summarizes regional shipment performance for the second quarter of 2024. Overall shipment volume grew 8.4% quarter over quarter, drive…
┃ [chunk 1] Regional performance is broken out in the table below. |**Region**|**Shipments**|**On-Time %**|**Revenue ($M)**| |---|---|---|---| |Northwest|128,400|97.2%|14.6| |West|204,910|98.1%|23.9| |South|176,2…
┃ [chunk 2] Action items: (1) complete the Northeast warehouse migration by July 15; (2) reallocate two delivery routes from Northwest to West to match demand; (3) review carrier contracts ahead of the Q3 peak se…


Compare the previews to Strategy 1: recursive splitting keeps paragraphs and the
table block more intact because it prefers paragraph/line boundaries over a hard
token count.

### Strategy 3 — Semantic chunking  *(splits at meaning boundaries)*

Instead of a fixed length, embed adjacent sentences and cut where the embedding
similarity drops — i.e. at genuine topic shifts. Higher quality on mixed content, at
the cost of running an embedding model during indexing.

**LangChain API:** `SemanticChunker` (from `langchain_experimental`) + embeddings

In [10]:
from langchain_experimental.text_splitter import SemanticChunker

semantic = SemanticChunker(embeddings, breakpoint_threshold_type="percentile")
sem_chunks = semantic.create_documents([contract])
inspect_chunks(sem_chunks)

→ 2 chunks | tokens min/avg/max = 31/141/252
┃ [chunk 0] Master Services Agreement This Master Services Agreement ("Agreement") is entered into between Northwind Logistics, Inc. ("Provider") and the Client, effective as of the date of last signature. 1. Sco…
┃ [chunk 1] 6. Limitation of Liability Neither party's aggregate liability shall exceed the fees paid under this Agreement in the twelve (12) months preceding the claim.


### Strategy 4 — Sentence-window retrieval  *(retrieve small, expand at generation)*

Index one sentence at a time for precise matching, but at answer time expand the hit
to its neighboring sentences so the model gets enough context.

**LangChain has no built-in sentence-window parser**, so we implement it manually:
embed each sentence, store its neighbor window in metadata, retrieve the best
sentence, then swap in the window. *(In Notebook 03 this is a one-liner —
`SentenceWindowNodeParser` — which is exactly why LlamaIndex shines for this.)*

In [11]:
from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore

# Clean sentence split: break on lines first (so headings don't glue onto the next
# sentence), then on sentence-ending punctuation.
sentences = []
for block in contract.split("\n"):
    block = block.strip()
    if not block:
        continue
    for s in re.split(r"(?<=[.!?])\s+", block):
        s = s.strip()
        if len(s) > 15:
            sentences.append(s)

WINDOW = 2  # neighbor sentences on each side
sentence_docs = []
for i, s in enumerate(sentences):
    lo, hi = max(0, i - WINDOW), min(len(sentences), i + WINDOW + 1)
    sentence_docs.append(Document(page_content=s, metadata={"window": " ".join(sentences[lo:hi]), "idx": i}))

sw_store = InMemoryVectorStore.from_documents(sentence_docs, embeddings)

query = "What is the home-internet stipend for remote employees?"
hit = sw_store.similarity_search(query, k=1)[0]
print("QUERY:", query)
print("\n-- Retrieved SENTENCE (what matched: small & precise) --\n", hit.page_content)
print("\n-- Expanded WINDOW (what the LLM actually gets) --\n", hit.metadata["window"])
print("\nNote: the matched sentence alone omits 'Contractors are not eligible' -- the window restores it.")

QUERY: What is the home-internet stipend for remote employees?

-- Retrieved SENTENCE (what matched: small & precise) --
 Employees working remotely at least three (3) days per week are eligible for a home-internet stipend of $45 per month, reimbursed via the monthly expense report.

-- Expanded WINDOW (what the LLM actually gets) --
 Provider may suspend services immediately for non-payment exceeding sixty (60) days. Remote-Work Reimbursement (HR-2024-17 §7.3) Employees working remotely at least three (3) days per week are eligible for a home-internet stipend of $45 per month, reimbursed via the monthly expense report. Contractors are not eligible. Limitation of Liability

Note: the matched sentence alone omits 'Contractors are not eligible' -- the window restores it.


### Strategy 5 — Parent-document / hierarchical (small-to-big)

Index small child chunks for precise matching, but return their larger **parent**
section to the model. You get the precision of small chunks with the context of large
ones.

**LangChain API:** `ParentDocumentRetriever` + a vector store (children) + a doc store
(parents). *(In Notebook 03: `HierarchicalNodeParser` + `AutoMergingRetriever`.)*

In [12]:
from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_core.stores import InMemoryStore

parent_splitter = RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap=0)
child_splitter  = RecursiveCharacterTextSplitter(chunk_size=150, chunk_overlap=0)

pdr = ParentDocumentRetriever(
    vectorstore=InMemoryVectorStore(embeddings),
    docstore=InMemoryStore(),
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)
pdr.add_documents([Document(page_content=contract)])

# The small child chunks that get embedded:
child_hits = pdr.vectorstore.similarity_search("remote work internet stipend", k=1)
print("── Matched CHILD chunk (small, precise) ──\n", child_hits[0].page_content, "\n")

# What the retriever actually returns — the bigger parent:
parent_docs = pdr.invoke("remote work internet stipend")
print("── Returned PARENT chunk (big, full context) ──\n", parent_docs[0].page_content[:400], "…")

── Matched CHILD chunk (small, precise) ──
 Employees working remotely at least three (3) days per week are eligible for a home-internet stipend of $45 per month, reimbursed via the monthly 



── Returned PARENT chunk (big, full context) ──
 Employees working remotely at least three (3) days per week are eligible for a home-internet stipend of $45 per month, reimbursed via the monthly expense report. Contractors are not eligible.

6. Limitation of Liability

Neither party's aggregate liability shall exceed the fees paid under this Agreement in the twelve (12) months preceding the claim. …


### Strategy 6 — Document-type-aware chunking

Different content wants different splitters: split **code** on function boundaries,
**Markdown** on headers, **HTML** on structure. One size does not fit all.

**LangChain APIs:** `RecursiveCharacterTextSplitter.from_language`,
`MarkdownHeaderTextSplitter`, `HTMLHeaderTextSplitter`

In [13]:
from langchain_text_splitters import (RecursiveCharacterTextSplitter, Language,
                                       MarkdownHeaderTextSplitter, HTMLHeaderTextSplitter)

# (a) Code — split on Python structure
code_sample = """
def ingest(path):
    text = load(path)
    return clean(text)

class Indexer:
    def __init__(self, store):
        self.store = store
    def add(self, chunks):
        self.store.upsert(chunks)
"""
py_splitter = RecursiveCharacterTextSplitter.from_language(Language.PYTHON, chunk_size=80, chunk_overlap=0)
print("CODE chunks:")
inspect_chunks(py_splitter.create_documents([code_sample]), n_preview=3, width=90)

# (b) Markdown — split on headers.
# NOTE: report_md only has ONE "#" heading and no "##" subheadings, so splitting it here
# would silently produce a single chunk and not demonstrate the splitter at all. We use a
# multi-section Markdown doc instead so the header-based split is actually visible.
markdown_doc = """# Support FAQ
## Shipping
Standard shipping takes 3-5 business days.
## Returns
Returns are accepted within 30 days of delivery.
## Tracking
Every shipment includes a tracking number."""
md_splitter = MarkdownHeaderTextSplitter([("#", "h1"), ("##", "h2")], strip_headers=False)
print("\nMARKDOWN chunks (split on # / ## headers):")
inspect_chunks(md_splitter.split_text(markdown_doc), n_preview=3, width=90)

# (c) HTML — split on structure
html_raw = (DATA / "sample_page.html").read_text(encoding="utf-8")
html_splitter = HTMLHeaderTextSplitter([("h1", "h1"), ("h2", "h2")])
print("\nHTML chunks (FAQ page):")
inspect_chunks(html_splitter.split_text(html_raw), n_preview=3, width=90)

CODE chunks:
→ 3 chunks | tokens min/avg/max = 14/15/18
┃ [chunk 0] def ingest(path): text = load(path) return clean(text)
┃ [chunk 1] class Indexer: def __init__(self, store): self.store = store
┃ [chunk 2] def add(self, chunks): self.store.upsert(chunks)

MARKDOWN chunks (split on # / ## headers):
→ 3 chunks | tokens min/avg/max = 10/13/17
┃ [chunk 0] # Support FAQ ## Shipping Standard shipping takes 3-5 business days.
┃ [chunk 1] ## Returns Returns are accepted within 30 days of delivery.
┃ [chunk 2] ## Tracking Every shipment includes a tracking number.

HTML chunks (FAQ page):
→ 8 chunks | tokens min/avg/max = 1/21/67
┃ [chunk 0] Northwind Logistics — Support FAQ
┃ [chunk 1] Answers to common questions about shipping, tracking, and returns.
┃ [chunk 2] Shipping


## 4. Summary — the six strategies in LangChain

| # | Strategy | LangChain API | When to use |
|---|----------|---------------|-------------|
| 1 | Fixed-size / token | `TokenTextSplitter` | Baseline; uniform text |
| 2 | Recursive | `RecursiveCharacterTextSplitter` | Sane default for general text |
| 3 | Semantic | `SemanticChunker` (experimental) | Mixed-density corpora |
| 4 | Sentence-window | *manual* (metadata window) | Precise match, need surrounding context |
| 5 | Parent / small-to-big | `ParentDocumentRetriever` | Multi-hop, long-document QA |
| 6 | Document-type-aware | `from_language`, `MarkdownHeaderTextSplitter`, `HTMLHeaderTextSplitter` | PDFs, code, tables, HTML |

### What we proved

- **Parsing sets the ceiling.** The table survived layout-aware parsing and was
  destroyed by naive extraction — before any chunking happened.
- **Chunking maturity is a ramp**, not a switch: start with recursive, escalate to
  semantic / sentence-window / small-to-big only when retrieval quality demands it.
- Strategies **4 and 5** needed extra wiring in LangChain (manual windows,
  `ParentDocumentRetriever`).

### A 7th technique worth knowing: Contextual Retrieval

All six strategies above decide *where* to cut. A complementary, increasingly
standard production technique decides *what to prepend* to each chunk before it's
embedded: Anthropic's **Contextual Retrieval** (2024) asks an LLM to generate a short
(50–100 token) description of how a chunk fits into the whole document — e.g. *"This
chunk is from Northwind's Q2 2024 report, discussing Northeast regional
performance"* — and prepends that to the chunk before embedding **and** before BM25
indexing. Reported results: ~35% fewer failed retrievals from context alone, ~49%
combined with contextual BM25, ~67% combined with reranking. It costs one extra LLM
call per chunk at *indexing* time (cheap and cacheable — it runs once, not per
query), and it composes with every strategy above rather than replacing any of them.

### Next

Open **Notebook 03** to see this *same* pipeline in **LlamaIndex**, where sentence-
window and hierarchical chunking are first-class node parsers. Comparing the two on
identical documents is the whole point — same concepts, different ergonomics.